# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Yaqoob/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring.**

The repo's own reference pipeline (`scripts/01-05`) already sketches this lane end-to-end — a
hand-written baseline rule, a trained model, and a ranked "fix this first" queue. I'm picking it
because it maps onto a problem every content team actually has: hundreds or thousands of pages,
one small team, and no way to read every page's trend line by hand. A model that ranks pages by
how worth-reviewing they are turns an unreadable spreadsheet into a short, ordered list. It also
gives me a strong baseline to beat honestly (Precision@50 ≈ 0.24 on the hand rule) rather than
inventing a scoring system from nothing.


In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("Rows, columns:", df.shape)
print(df["trend_direction"].value_counts())


Rows, columns: (30000, 44)
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. The question: decision, action, cost of a wrong call

**Decision:** given a fixed content-review budget (say, the top 50-100 pages a team can
realistically look at this sprint), which pages should go on that list first?

**Who acts on it:** a content/SEO lead deciding where to spend limited writer and editor time —
rewrite, refresh, or leave alone.

**Cost of a wrong call:** two kinds of mistake, and they cost differently.
- *False positive* (flagged as declining/worth-reviewing, but it wasn't): wastes an editor's
  afternoon on a page that didn't need it — annoying, low stakes.
- *False negative* (a genuinely declining, high-impression page never surfaces): the page keeps
  losing visibility silently until someone stumbles on it much later — the expensive mistake,
  because traffic already lost is harder to win back than traffic that's merely flat.

That asymmetry is why the model should be judged on **Precision@K** on a fixed-size review list
(can the top of the ranking be trusted), not on raw accuracy across all 30,000 pages.


In [2]:
declining = df[df["trend_direction"] == "down"]
measurable = df[(df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)]
declining_measurable = declining[
    (declining["impressions_90d"] >= 100) & (declining["sessions_90d"] > 0)
]

print(f"Declining pages: {len(declining):,} ({len(declining)/len(df)*100:.1f}% of all pages)")
print(f"Measurable-opportunity pages (>=100 impr, has sessions): {len(measurable):,} "
      f"({len(measurable)/len(df)*100:.1f}%)")
print(f"Declining AND measurable: {len(declining_measurable):,} "
      f"({len(declining_measurable)/len(df)*100:.1f}%)")

impressions_lost_30d = (declining["impressions_prev_30d"] - declining["impressions_last_30d"]).sum()
print(f"\nImpressions lost per 30d window, summed across declining pages: {impressions_lost_30d:,.0f}")


Declining pages: 16,262 (54.2% of all pages)
Measurable-opportunity pages (>=100 impr, has sessions): 22,006 (73.4%)
Declining AND measurable: 13,152 (43.8%)

Impressions lost per 30d window, summed across declining pages: 14,096,894


## 3. Quick look at the data (2-3 real numbers)

- **54.2%** of the 30,000 pages in this slice are labeled `trend_direction == "down"`
  (16,262 pages) — a majority-declining dataset, so a "flag everything" baseline would already
  score well on accuracy but be useless as a review list. This is exactly why Precision@K on a
  fixed-size list is the right metric, not accuracy.
- Of those declining pages, **43.8%** (13,152) also clear the `measurable_opportunity` bar
  (≥100 impressions/90d and at least one session) — meaning the decline is happening on pages
  that actually have traffic worth protecting, not statistical noise on near-zero-impression
  pages.
- Declining pages alone are losing roughly **14.1 million impressions per 30-day window**
  (summed across all of them, last-30 vs prev-30) — a rough proxy for how much visibility is up
  for grabs if the highest-value declines get reviewed first.

Together these numbers say: the problem is real, it's concentrated (not spread evenly), and
there's a clear "worth it" subset to prioritize — which is exactly what a ranked action engine
is for.


In [3]:
content_type_counts = df["content_type"].value_counts()
client_count = df["client_id"].nunique()

print("Distinct clients in this slice:", client_count)
print("\nContent type mix:")
print(content_type_counts)

print("\nMedian impressions_90d, declining vs non-declining:")
print(df.groupby(df["trend_direction"] == "down")["impressions_90d"].median())


Distinct clients in this slice: 32

Content type mix:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

Median impressions_90d, declining vs non-declining:
trend_direction
False    472.0
True     961.0
Name: impressions_90d, dtype: float64


## 4. Careful words: what I can and can't claim

**What this work will be able to say:**
- Observed: which pages in this dataset showed a measured decline in search impressions over
  the trailing 30-day window versus the prior 30-day window.
- Directional: which content and engagement signals are *associated with* being flagged as
  declining, based on a model trained and validated on held-out clients.
- Decision-support: a ranked list, with reason codes, of which pages are the best use of a
  limited review budget this sprint — a recommendation, not an instruction.

**What it will never say:**
- No causal claim that any single factor *caused* a page's ranking to change — this is
  observational data, not an experiment.
- No claim to predict or explain Google's ranking algorithm itself. `trend_direction` and
  `trend_pct` describe *this site's own* impression trend, not a search-engine mechanism.
- No claim of a guaranteed traffic outcome if a flagged page is refreshed — the model estimates
  review-worthiness, not the result of acting on it.


In [4]:
leakage_prone_columns = ["trend_direction", "trend_pct"]
print("Columns that must NEVER be model features (they define the label):")
for c in leakage_prone_columns:
    print(" -", c)

# Sanity check: these are exactly the columns used to build the label
is_declining_label = (df["trend_direction"] == "down").astype(int)
print("\nis_declining_label matches trend_direction==down for all rows:",
      (is_declining_label == (df["trend_direction"] == "down").astype(int)).all())


Columns that must NEVER be model features (they define the label):
 - trend_direction
 - trend_pct

is_declining_label matches trend_direction==down for all rows: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.